# 02 Reranking Demo

Run the full two-stage flow and inspect per-paper criterion scores and final ranking.

## What This Notebook Does
This notebook runs and inspects the full two-stage demo pipeline.

High-level flow:
1. Run the packaged end-to-end pipeline (`run_demo`) to produce output rows.
2. Re-run one query step-by-step to inspect intermediate score components.
3. Compare semantic + recency criteria and inspect final weighted rank order.
4. Optionally export a notebook-specific reranking preview CSV.
        

### Step 1: Notebook Runtime Setup
This cell ensures project imports work by placing repo root on `sys.path`.
        

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")

Project root: /Users/juno/paper-reranking-demo


### Step 2: Import End-to-End and Scoring Utilities
This cell imports pipeline orchestration plus lower-level scoring helpers for detailed inspection.
        

In [2]:
import pandas as pd

from src import config
from src.pipeline.run_demo import run_demo
from src.schema.query_instantiator import load_query_specs
from src.retrieval.semantic_scholar_client import SemanticScholarClient
from src.retrieval.candidate_retriever import CandidateRetriever
from src.scoring.semantic_similarity import compute_semantic_scores, SEMANTIC_SLOT_NAMES
from src.scoring.recency import score_recency_for_papers
from src.scoring.weighted_ranker import rank_papers

/Users/juno/paper-reranking-demo/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 3: Run the Full Demo Pipeline
This cell executes `run_demo(...)` across all configured queries and shows the resulting output table.
        

In [3]:
results_df = run_demo(
    query_path=config.DEFAULT_QUERY_FILE,
    output_path=config.DEFAULT_OUTPUT_FILE,
    top_k=config.TOP_K,
)

print(f"Rows produced: {len(results_df)}")
results_df.head(20)

2026-03-23 11:41:50 | INFO | src.pipeline.run_demo | Running query: Graph neural networks for molecular property prediction
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6918.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-23 11:41:54 | INFO | src.pipeline.run_demo | Running query: RAG methods for scientific question answering
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2663.99it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identi

Rows produced: 60


,raw_query,paper_id,title,year,final_score,topic_match,method_match,relationship_match,recency
0,Graph neural networks for molecular property p...,8495965cf4ff5eb8010c7f126690106c135f23ab,Kolmogorov–Arnold graph neural networks for mo...,2025.0,0.693880,0.896650,0.460850,0.633858,0.833333
1,Graph neural networks for molecular property p...,82a4356ab71189ae9837b7752ba31fa570a04514,Composite Graph Neural Networks for Molecular ...,2024.0,0.673873,0.828191,0.500581,0.668663,0.666667
2,Graph neural networks for molecular property p...,c70d99bc680615b38111f949ce1ff0cc171d3967,Cross-dependent graph neural networks for mole...,2022.0,0.673188,0.893761,0.538622,0.661810,0.333333
3,Graph neural networks for molecular property p...,0c11d5fd81ba37a97ed96c4091f1996e7e5ec709,Impact of Local Descriptors Derived from Machi...,2026.0,0.645404,0.758618,0.436814,0.595373,1.000000
4,Graph neural networks for molecular property p...,c639624193847693bb2fd4b1319ca3229093c412,Mfgnn: Multi‐Scale Feature‐Attentive Graph Neu...,2025.0,0.630871,0.702769,0.470157,0.642085,0.833333
5,Graph neural networks for molecular property p...,6039ef4676d535ef5b60f6315036a84b9d5ebe3b,Chain-aware graph neural networks for molecula...,2024.0,0.629132,0.721735,0.530521,0.602807,0.666667
6,Graph neural networks for molecular property p...,f35480e309f9cdaab6ecc087fdd29d448b330d03,Bayesian Graph Neural Networks for Molecular P...,2020.0,0.624492,0.891636,0.488654,0.663291,0.000000
7,Graph neural networks for molecular property p...,2b2e2d05489ee9a50641226d0f68d6b08bb09b62,KA-GNN: Kolmogorov-Arnold Graph Neural Network...,2024.0,0.610412,0.723380,0.534578,0.520755,0.666667
8,Graph neural networks for molecular property p...,9c575c388947e9ff37f6023853c83421c1a3614b,Comparison of Atom Representations in Graph Ne...,2020.0,0.608227,0.787421,0.498700,0.732079,0.000000
9,Graph neural networks for molecular property p...,22ae90261ccd2a124c46f694ccced26cf1164a04,Extended study on atomic featurization in grap...,2023.0,0.605744,0.721273,0.486476,0.629422,0.500000


### Step 4: Select One Query and Retrieve Candidates
This cell prepares a focused single-query run so intermediate scoring behavior can be examined transparently.
        

In [4]:
query_specs = load_query_specs(config.DEFAULT_QUERY_FILE)
query_spec = query_specs[0]
TOP_K = 10

client = SemanticScholarClient(api_key=config.SEMANTIC_SCHOLAR_API_KEY)
retriever = CandidateRetriever(client=client)
candidates = retriever.retrieve_candidates(raw_query=query_spec.raw_query, top_k=TOP_K)

print(f"Query: {query_spec.raw_query}")
print(f"Candidates: {len(candidates)}")

Query: Graph neural networks for molecular property prediction
Candidates: 10


### Step 5: Compute Criterion Scores and Rerank
This cell computes semantic criterion scores, computes recency, combines them with weights, and produces a ranked DataFrame.

Why this is after `run_demo`: `run_demo` gives batch output; this step gives detailed per-query diagnostics.
        

In [5]:
if candidates:
    try:
        semantic_scores = compute_semantic_scores(
            papers=candidates,
            query_spec=query_spec,
            embedder=None,
            slot_names=SEMANTIC_SLOT_NAMES,
        )
    except Exception as exc:
        print(f"Semantic scoring failed; falling back to zeros: {exc}")
        semantic_scores = {
            p.paper_id: {slot: 0.0 for slot in SEMANTIC_SLOT_NAMES}
            for p in candidates
        }

    recency_scores = score_recency_for_papers(candidates)

    criterion_scores_by_paper = {}
    for p in candidates:
        s = dict(semantic_scores.get(p.paper_id, {}))
        s["recency"] = recency_scores.get(p.paper_id, 0.0)
        criterion_scores_by_paper[p.paper_id] = s

    ranked = rank_papers(
        papers=candidates,
        criterion_scores_by_paper=criterion_scores_by_paper,
        weights=config.DEFAULT_WEIGHTS,
    )

    reranked_rows = []
    for idx, item in enumerate(ranked, start=1):
        row = {
            "reranked_position": idx,
            "paper_id": item.paper.paper_id,
            "title": item.paper.title,
            "year": item.paper.year,
            "final_score": item.final_score,
        }
        row.update(item.criterion_scores)
        reranked_rows.append(row)

    reranked_df = pd.DataFrame(reranked_rows)
    reranked_df.head(20)
else:
    print("No candidates to rerank. Check API/network settings.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 30811.88it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Step 6: Save Notebook Preview Output
This cell saves a notebook-specific reranking preview file if results exist.
        

In [6]:
if 'reranked_df' in globals() and not reranked_df.empty:
    out_path = ROOT / "data" / "outputs" / "notebook_reranked_preview.csv"
    reranked_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

Saved: /Users/juno/paper-reranking-demo/data/outputs/notebook_reranked_preview.csv
